# 绘制全部 `XRS_analysis_all_data.txt`

递归查找项目中的所有 `XRS_analysis_all_data.txt`，读取第一列作为能量转移。每个通道使用独立子图，并按 5×5 布局分页绘制。q 值从数据文件同目录的 `qc_roi_fits.tsv` 读取并显示在子图标题中。

In [ ]:
from pathlib import Path
import re

from xrsabre.paths import load_workspace

import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "lines.linewidth": 1.1,
})

FILE_NAME = "XRS_analysis_all_data.txt"
SAVE_FIGURES = False  # 改为 True 可保存到配置的 diagnostics/xrs_plots

workspace = load_workspace()
reduced_root = workspace.reduced
data_files = sorted(
    path for path in reduced_root.rglob(FILE_NAME)
    if ".pixi" not in path.parts and ".git" not in path.parts
)

if not data_files:
    raise FileNotFoundError(f"在 {reduced_root} 下没有找到 {FILE_NAME}")

print(f"数据目录：{reduced_root}")
print(f"找到 {len(data_files)} 个数据文件：")
for path in data_files:
    print(f"  - {path.relative_to(reduced_root)}")

In [ ]:
def load_xrs_data(path):
    """读取一个 XRS 文本文件，并检查/清理数值列。"""
    frame = pd.read_csv(path, sep="\t")
    if frame.shape[1] < 2:
        raise ValueError(f"{path} 至少需要一个横坐标列和一个信号列")

    frame = frame.apply(pd.to_numeric, errors="coerce")
    x_name = frame.columns[0]
    frame = frame.dropna(subset=[x_name]).sort_values(x_name)
    signal_columns = [name for name in frame.columns[1:] if frame[name].notna().any()]
    if not signal_columns:
        raise ValueError(f"{path} 中没有可绘制的信号列")
    return frame, x_name, signal_columns


def load_q_values(data_path):
    """从数据文件旁的 ROI/QC 表读取 {通道名: q_ave}。"""
    preferred = data_path.parent / "qc_roi_fits.tsv"
    candidates = [preferred] if preferred.exists() else sorted(data_path.parent.glob("*roi*.tsv"))
    if not candidates:
        return {}, None

    q_path = candidates[0]
    metadata = pd.read_csv(q_path, sep="\t")
    if "q_ave" not in metadata.columns:
        return {}, q_path

    id_column = next((name for name in ("roi_id", "crystal") if name in metadata.columns), None)
    if id_column is None:
        return {}, q_path

    q_values = pd.to_numeric(metadata["q_ave"], errors="coerce")
    return dict(zip(metadata[id_column].astype(str), q_values)), q_path


def safe_name(text):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", text).strip("_") or "xrs_plot"

## 数据概览

In [ ]:
summary = []
for path in data_files:
    frame, x_name, signal_columns = load_xrs_data(path)
    q_values, q_path = load_q_values(path)
    summary.append({
        "file": str(path.relative_to(reduced_root)),
        "rows": len(frame),
        "channels": len(signal_columns),
        "q_metadata": str(q_path.relative_to(reduced_root)) if q_path else "未找到",
        "q_matched": sum(pd.notna(q_values.get(name)) for name in signal_columns),
        "x_column": x_name,
        "x_min": frame[x_name].min(),
        "x_max": frame[x_name].max(),
    })

pd.DataFrame(summary)

## 以 5×5 布局绘制独立通道

每个通道占一个独立子图，每页最多显示 25 个通道。子图标题显示通道名及其平均动量转移 `q_ave`（Å⁻¹）；超过 25 个通道时自动生成下一页。

In [ ]:
output_root = workspace.diagnostics / "xrs_plots"

for path in data_files:
    frame, x_name, signal_columns = load_xrs_data(path)
    q_values, q_path = load_q_values(path)
    relative_path = path.relative_to(reduced_root)
    output_dir = output_root / relative_path.parent
    if SAVE_FIGURES:
        output_dir.mkdir(parents=True, exist_ok=True)

    page_size = 25
    page_count = (len(signal_columns) + page_size - 1) // page_size
    for page_index, start in enumerate(range(0, len(signal_columns), page_size), start=1):
        page_columns = signal_columns[start:start + page_size]
        fig, axes = plt.subplots(5, 5, figsize=(18, 15), sharex=True, squeeze=False)

        for ax, column in zip(axes.flat, page_columns):
            q_value = q_values.get(column, float("nan"))
            q_text = f"q={q_value:.3f} Å⁻¹" if pd.notna(q_value) else "q=N/A"
            ax.plot(frame[x_name], frame[column], color="#165a72", linewidth=1.0)
            ax.set_title(f"{column}\n{q_text}", fontsize=9)
            ax.tick_params(labelsize=7)

        for ax in axes.flat[len(page_columns):]:
            ax.set_visible(False)

        fig.supxlabel(x_name, fontsize=11)
        fig.supylabel("Intensity (a.u.)", fontsize=11)
        fig.suptitle(
            f"{relative_path}  |  page {page_index}/{page_count}",
            fontsize=14,
        )
        fig.tight_layout(rect=(0.02, 0.02, 1, 0.97))

        if SAVE_FIGURES:
            fig.savefig(
                output_dir / f"channels_page_{page_index:02d}.png",
                dpi=200, bbox_inches="tight",
            )

        plt.show()
        plt.close(fig)

if SAVE_FIGURES:
    print(f"图片已保存到：{output_root}")